# TBGuard Notebook 3 — WHO Mutation Catalogue Genomic/DNA Expert


In [1]:
!pip install -q pandas numpy joblib openpyxl

In [2]:
import os, json, re, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

BASE_DIR = Path('tbguard_project')
DATA_DIR = BASE_DIR / 'data'
GENOMIC_DIR = DATA_DIR / 'genomic'
MODEL_DIR = BASE_DIR / 'artifacts' / 'models'
GENOMIC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Genomic dir:', GENOMIC_DIR.resolve())
print('Model dir:', MODEL_DIR.resolve())

Genomic dir: /content/tbguard_project/data/genomic
Model dir: /content/tbguard_project/artifacts/models


## 1. Upload WHO catalogue Excel file

Upload `WHO-UCN-TB-2023.7-eng.xlsx`.

In [3]:
try:
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded.keys():
        target = GENOMIC_DIR / 'WHO-UCN-TB-2023.7-eng.xlsx'
        shutil.move(filename, target)
        print('Saved upload to:', target)
        break
except Exception as e:
    print('Upload skipped or not running in Colab:', e)

who_path = GENOMIC_DIR / 'WHO-UCN-TB-2023.7-eng.xlsx'
if not who_path.exists():
    raise FileNotFoundError('WHO-UCN-TB-2023.7-eng.xlsx not found. Upload it or set who_path manually.')
print('Using:', who_path)

Saving WHO-UCN-TB-2023.7-eng.xlsx to WHO-UCN-TB-2023.7-eng.xlsx
Saved upload to: tbguard_project/data/genomic/WHO-UCN-TB-2023.7-eng.xlsx
Using: tbguard_project/data/genomic/WHO-UCN-TB-2023.7-eng.xlsx


## 2. Parse WHO catalogue

The catalogue sheet has two top rows before the actual headers, so we set the header from row 3.

In [5]:
raw = pd.read_excel(who_path, sheet_name='Catalogue_master_file', header=None)
headers = raw.iloc[2].astype(str).tolist()
catalogue = raw.iloc[3:].copy()
catalogue.columns = headers
catalogue = catalogue.loc[:, ~catalogue.columns.duplicated()].copy()

keep_cols = ['drug', 'gene', 'mutation', 'variant', 'tier', 'effect', 'FINAL CONFIDENCE GRADING', 'Comment']
keep_cols = [c for c in keep_cols if c in catalogue.columns]
catalogue = catalogue[keep_cols].copy()

for col in catalogue.columns:
    catalogue[col] = catalogue[col].astype(str).replace({'nan': np.nan})

catalogue = catalogue.dropna(subset=['drug','gene','variant','FINAL CONFIDENCE GRADING'])
print('Catalogue shape:', catalogue.shape)
print('Columns:', catalogue.columns.tolist())
display(catalogue.head())
print('\nFinal grading distribution:')
print(catalogue['FINAL CONFIDENCE GRADING'].value_counts())
print('\nDrug counts:')
print(catalogue['drug'].value_counts().head(20))

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Catalogue shape: (48152, 8)
Columns: ['drug', 'gene', 'mutation', 'variant', 'tier', 'effect', 'FINAL CONFIDENCE GRADING', 'Comment']


,drug,gene,mutation,variant,tier,effect,FINAL CONFIDENCE GRADING,Comment
3,Amikacin,bacA,c.102G>A,bacA_c.102G>A,2,synonymous_variant,4) Not assoc w R - Interim,NaN
4,Amikacin,bacA,c.1044G>A,bacA_c.1044G>A,2,synonymous_variant,5) Not assoc w R,NaN
5,Amikacin,bacA,c.105C>G,bacA_c.105C>G,2,synonymous_variant,4) Not assoc w R - Interim,NaN
6,Amikacin,bacA,c.1065T>G,bacA_c.1065T>G,2,synonymous_variant,4) Not assoc w R - Interim,NaN
7,Amikacin,bacA,c.1080G>A,bacA_c.1080G>A,2,synonymous_variant,4) Not assoc w R - Interim,NaN



Final grading distribution:
FINAL CONFIDENCE GRADING
3) Uncertain significance     33906
4) Not assoc w R - Interim    12379
2) Assoc w R - Interim         1130
5) Not assoc w R                484
1) Assoc w R                    253
Name: count, dtype: int64

Drug counts:
drug
Isoniazid       7286
Rifampicin      7274
Ethambutol      7118
Streptomycin    3061
Levofloxacin    3047
Ethionamide     2748
Moxifloxacin    2723
Capreomycin     2654
Pyrazinamide    2572
Kanamycin       2240
Amikacin        2190
Clofazimine     1903
Bedaquiline     1465
Delamanid        943
Linezolid        928
Name: count, dtype: int64


## 3. Normalize resistance grading

In [6]:
def normalize_confidence_grade(x):
    x = str(x)
    if x.startswith('1)'):
        return 'associated_with_resistance'
    if x.startswith('2)'):
        return 'associated_with_resistance_interim'
    if x.startswith('3)'):
        return 'uncertain_significance'
    if x.startswith('4)'):
        return 'not_associated_interim'
    if x.startswith('5)'):
        return 'not_associated'
    return 'unknown'

catalogue['resistance_grade_normalized'] = catalogue['FINAL CONFIDENCE GRADING'].apply(normalize_confidence_grade)
catalogue['is_resistance_associated'] = catalogue['resistance_grade_normalized'].isin([
    'associated_with_resistance',
    'associated_with_resistance_interim'
]).astype(int)

important_genes = ['rpoB', 'katG', 'inhA', 'gyrA', 'gyrB', 'rrs', 'eis', 'embB', 'pncA']
important_subset = catalogue[catalogue['gene'].isin(important_genes)].copy()
print('Important-gene subset:', important_subset.shape)
print(important_subset.groupby(['drug','gene','resistance_grade_normalized']).size().head(30))

Important-gene subset: (11138, 10)
drug         gene  resistance_grade_normalized       
Amikacin     eis   associated_with_resistance_interim       1
                   not_associated                           3
                   not_associated_interim                  65
                   uncertain_significance                 187
             rrs   associated_with_resistance               2
                   associated_with_resistance_interim       1
                   not_associated                          52
                   uncertain_significance                 611
Capreomycin  rrs   associated_with_resistance               3
                   not_associated                          24
                   uncertain_significance                 603
Ethambutol   embB  associated_with_resistance              13
                   not_associated                          12
                   not_associated_interim                 385
                   uncertain_significance  

## 4. Save cleaned WHO catalogue CSV for RAG and traceability

In [7]:
clean_path = GENOMIC_DIR / 'who_mutation_catalogue_cleaned.csv'
important_path = GENOMIC_DIR / 'who_mutation_catalogue_important_genes.csv'
catalogue.to_csv(clean_path, index=False)
important_subset.to_csv(important_path, index=False)
print('Saved:', clean_path)
print('Saved:', important_path)

Saved: tbguard_project/data/genomic/who_mutation_catalogue_cleaned.csv
Saved: tbguard_project/data/genomic/who_mutation_catalogue_important_genes.csv


## 5. Build WHO-rule genomic expert artifact

The app will accept DNA marker inputs such as `rpoB=1`, `katG=1`, `gyrA=1` and classify drug-resistance concern.

In [12]:
rule_map = {
    'rpoB': {
        'drug': 'Rifampicin',
        'interpretation': 'rpoB mutation marker suggests rifampicin resistance concern.',
        'clinical_role': 'Rifampicin resistance is a major marker for RR-TB and possible MDR-TB evaluation.'
    },
    'katG': {
        'drug': 'Isoniazid',
        'interpretation': 'katG mutation marker suggests isoniazid resistance concern.',
        'clinical_role': 'Isoniazid resistance combined with rifampicin resistance supports MDR-TB concern.'
    },
    'inhA': {
        'drug': 'Isoniazid',
        'interpretation': 'inhA mutation marker suggests isoniazid resistance concern.',
        'clinical_role': 'inhA promoter or gene-associated resistance can contribute to isoniazid resistance.'
    },
    'gyrA': {
        'drug': 'Fluoroquinolones',
        'interpretation': 'gyrA mutation marker suggests fluoroquinolone resistance concern.',
        'clinical_role': 'Fluoroquinolone resistance in MDR/RR-TB raises pre-XDR concern.'
    },
    'gyrB': {
        'drug': 'Fluoroquinolones',
        'interpretation': 'gyrB mutation marker suggests fluoroquinolone resistance concern.',
        'clinical_role': 'Fluoroquinolone resistance in MDR/RR-TB raises pre-XDR concern.'
    },
    'rrs': {
        'drug': 'Second-line injectable drugs',
        'interpretation': 'rrs mutation marker suggests second-line injectable resistance concern.',
        'clinical_role': 'Second-line resistance requires specialist review and drug susceptibility testing.'
    },
    'eis': {
        'drug': 'Kanamycin / injectable-associated resistance',
        'interpretation': 'eis mutation marker suggests injectable-associated resistance concern.',
        'clinical_role': 'Injectable-associated resistance should be interpreted with confirmatory testing.'
    },
    'embB': {
        'drug': 'Ethambutol',
        'interpretation': 'embB mutation marker suggests ethambutol resistance concern.',
        'clinical_role': 'Ethambutol resistance may affect first-line treatment selection.'
    },
    'pncA': {
        'drug': 'Pyrazinamide',
        'interpretation': 'pncA mutation marker suggests pyrazinamide resistance concern.',
        'clinical_role': 'Pyrazinamide resistance may affect first-line treatment selection.'
    }
}

catalogue_summary = {}

for gene in rule_map:
    sub = catalogue[catalogue['gene'] == gene]
    catalogue_summary[gene] = {
        'num_catalogue_rows': int(len(sub)),
        'num_resistance_associated_rows': int(sub['is_resistance_associated'].sum()),
        'drugs_in_catalogue': sorted(sub['drug'].dropna().astype(str).unique().tolist())
    }

artifact = {
    'model_type': 'who_rule_based_catalogue_expert',
    'model_name': 'WHO-catalogue-style genomic resistance expert',
    'features': list(rule_map.keys()),
    'rule_map': rule_map,
    'catalogue_summary': catalogue_summary,
    'catalogue_cleaned_csv': str(clean_path),
    'important_genes_csv': str(important_path),
    'class_mapping': {
        0: 'No major resistance marker detected',
        1: 'Drug-resistant TB / MDR-TB concern',
        2: 'pre-XDR concern / MDR-TB with fluoroquinolone resistance'
    },
    'source': (
        'WHO Catalogue of mutations in Mycobacterium tuberculosis complex '
        'and their association with drug resistance, 2023 update.'
    ),
    'probability_note': (
        'This artifact is rule-based. Returned class scores are heuristic confidence scores, '
        'not calibrated ML probabilities.'
    )
}

artifact_path = MODEL_DIR / 'genomic_resistance_model.joblib'
joblib.dump(artifact, artifact_path)
print('Saved:', artifact_path)

metadata_path = MODEL_DIR / 'genomic_resistance_model_metadata.json'
metadata_safe = {k: v for k, v in artifact.items() if k != 'rule_map'}
metadata_path.write_text(json.dumps(metadata_safe, indent=2), encoding='utf-8')
print('Saved metadata:', metadata_path)

print(json.dumps(metadata_safe, indent=2)[:2000])

Saved: tbguard_project/artifacts/models/genomic_resistance_model.joblib
Saved metadata: tbguard_project/artifacts/models/genomic_resistance_model_metadata.json
{
  "model_type": "who_rule_based_catalogue_expert",
  "model_name": "WHO-catalogue-style genomic resistance expert",
  "features": [
    "rpoB",
    "katG",
    "inhA",
    "gyrA",
    "gyrB",
    "rrs",
    "eis",
    "embB",
    "pncA"
  ],
  "catalogue_summary": {
    "rpoB": {
      "num_catalogue_rows": 1502,
      "num_resistance_associated_rows": 136,
      "drugs_in_catalogue": [
        "Rifampicin"
      ]
    },
    "katG": {
      "num_catalogue_rows": 1527,
      "num_resistance_associated_rows": 135,
      "drugs_in_catalogue": [
        "Isoniazid"
      ]
    },
    "inhA": {
      "num_catalogue_rows": 520,
      "num_resistance_associated_rows": 16,
      "drugs_in_catalogue": [
        "Ethionamide",
        "Isoniazid"
      ]
    },
    "gyrA": {
      "num_catalogue_rows": 1587,
      "num_resistance_assoc

## 6. Inference function for Notebook 4 / Gradio

In [13]:
def load_who_genomic_artifact(path=MODEL_DIR / 'genomic_resistance_model.joblib'):
    return joblib.load(path)


def predict_genomic_resistance_who_rules(
    genomic_data,
    artifact_path=MODEL_DIR / 'genomic_resistance_model.joblib'
):
    """
    WHO-catalogue-style rule-based genomic resistance expert.

    This is not a trained ML classifier. It interprets common TB drug-resistance
    marker genes using WHO-catalogue-style logic and returns a structured output
    compatible with the LangGraph/Gradio app.
    """

    art = load_who_genomic_artifact(artifact_path)

    rule_map = art.get('rule_map', {})
    features = art.get(
        'features',
        ['rpoB', 'katG', 'inhA', 'gyrA', 'gyrB', 'rrs', 'eis', 'embB', 'pncA']
    )

    fallback_rule_map = {
        'rpoB': {
            'drug': 'Rifampicin',
            'interpretation': 'rpoB mutation marker suggests rifampicin resistance concern.',
            'clinical_role': 'Rifampicin resistance is a major marker for RR-TB and possible MDR-TB evaluation.'
        },
        'katG': {
            'drug': 'Isoniazid',
            'interpretation': 'katG mutation marker suggests isoniazid resistance concern.',
            'clinical_role': 'Isoniazid resistance combined with rifampicin resistance supports MDR-TB concern.'
        },
        'inhA': {
            'drug': 'Isoniazid',
            'interpretation': 'inhA mutation marker suggests isoniazid resistance concern.',
            'clinical_role': 'inhA promoter or gene-associated resistance can contribute to isoniazid resistance.'
        },
        'gyrA': {
            'drug': 'Fluoroquinolones',
            'interpretation': 'gyrA mutation marker suggests fluoroquinolone resistance concern.',
            'clinical_role': 'Fluoroquinolone resistance in MDR/RR-TB raises pre-XDR concern.'
        },
        'gyrB': {
            'drug': 'Fluoroquinolones',
            'interpretation': 'gyrB mutation marker suggests fluoroquinolone resistance concern.',
            'clinical_role': 'Fluoroquinolone resistance in MDR/RR-TB raises pre-XDR concern.'
        },
        'rrs': {
            'drug': 'Second-line injectable drugs',
            'interpretation': 'rrs mutation marker suggests second-line injectable resistance concern.',
            'clinical_role': 'Second-line resistance requires specialist review and drug susceptibility testing.'
        },
        'eis': {
            'drug': 'Kanamycin / injectable-associated resistance',
            'interpretation': 'eis mutation marker suggests injectable-associated resistance concern.',
            'clinical_role': 'Injectable-associated resistance should be interpreted with confirmatory testing.'
        },
        'embB': {
            'drug': 'Ethambutol',
            'interpretation': 'embB mutation marker suggests ethambutol resistance concern.',
            'clinical_role': 'Ethambutol resistance may affect first-line treatment selection.'
        },
        'pncA': {
            'drug': 'Pyrazinamide',
            'interpretation': 'pncA mutation marker suggests pyrazinamide resistance concern.',
            'clinical_role': 'Pyrazinamide resistance may affect first-line treatment selection.'
        }
    }

    markers = {}

    for gene in features:
        value = genomic_data.get(gene, 0)

        try:
            markers[gene] = int(value)
        except Exception:
            markers[gene] = 0

    rif_resistance = markers.get('rpoB', 0) == 1
    inh_resistance = markers.get('katG', 0) == 1 or markers.get('inhA', 0) == 1
    fluoroquinolone_resistance = markers.get('gyrA', 0) == 1 or markers.get('gyrB', 0) == 1
    injectable_resistance = markers.get('rrs', 0) == 1 or markers.get('eis', 0) == 1
    other_resistance = markers.get('embB', 0) == 1 or markers.get('pncA', 0) == 1

    markers_detected = [gene for gene, value in markers.items() if value == 1]

    resistance_evidence = []

    for gene in markers_detected:
        info = rule_map.get(gene, {})

        if not info or not info.get('drug') or not info.get('interpretation'):
            info = fallback_rule_map.get(gene, {})

        resistance_evidence.append({
            'gene': gene,
            'drug': info.get('drug', 'Unknown'),
            'interpretation': info.get(
                'interpretation',
                f'{gene} marker detected; resistance relevance should be reviewed.'
            ),
            'clinical_role': info.get(
                'clinical_role',
                'Requires clinical interpretation and confirmatory drug susceptibility testing.'
            )
        })

    if rif_resistance and inh_resistance and fluoroquinolone_resistance:
        class_id = 2
        resistance_class = 'Possible pre-XDR TB concern / MDR-TB with fluoroquinolone resistance'
        confidence = 'high'
    elif rif_resistance and inh_resistance:
        class_id = 1
        resistance_class = 'Possible MDR-TB concern'
        confidence = 'high'
    elif rif_resistance:
        class_id = 1
        resistance_class = 'Rifampicin-resistant TB concern; evaluate for MDR-TB'
        confidence = 'medium'
    elif inh_resistance or fluoroquinolone_resistance or injectable_resistance or other_resistance:
        class_id = 1
        resistance_class = 'Possible drug-resistance concern'
        confidence = 'medium'
    else:
        class_id = 0
        resistance_class = 'No major genomic resistance marker detected'
        confidence = 'medium'

    class_mapping = art.get(
        'class_mapping',
        {
            0: 'No major resistance marker detected',
            1: 'Drug-resistant TB / MDR-TB concern',
            2: 'pre-XDR concern / MDR-TB with fluoroquinolone resistance'
        }
    )

    class_probabilities = {
        class_mapping[0]: 0.85 if class_id == 0 else 0.10,
        class_mapping[1]: 0.85 if class_id == 1 else 0.10,
        class_mapping[2]: 0.85 if class_id == 2 else 0.10
    }

    return {
        'model_name': art.get('model_name', 'WHO-catalogue-style genomic resistance expert'),
        'model_type': art.get('model_type', 'rule_based_interpretable_genomic_expert'),
        'predicted_class_id': class_id,
        'resistance_class': resistance_class,
        'confidence': confidence,
        'markers_detected': markers_detected,
        'resistance_evidence': resistance_evidence,
        'interpreted_resistance': {
            'rifampicin': rif_resistance,
            'isoniazid': inh_resistance,
            'fluoroquinolone': fluoroquinolone_resistance,
            'second_line_injectable': injectable_resistance,
            'ethambutol_or_pyrazinamide': other_resistance
        },
        'class_probabilities': class_probabilities,
        'probability_note': art.get(
            'probability_note',
            'These are rule-based confidence scores, not calibrated ML probabilities.'
        ),
        'source_basis': art.get(
            'source',
            'WHO mutation catalogue concept: mutation markers interpreted as evidence for anti-TB drug resistance.'
        ),
        'human_in_loop_note': (
            'Genomic resistance interpretation must be confirmed by qualified clinicians '
            'and laboratory drug-susceptibility testing.'
        )
    }


sample_no_resistance = {
    'rpoB': 0,
    'katG': 0,
    'inhA': 0,
    'gyrA': 0,
    'gyrB': 0,
    'rrs': 0,
    'eis': 0,
    'embB': 0,
    'pncA': 0
}

sample_mdr = {
    'rpoB': 1,
    'katG': 1,
    'inhA': 0,
    'gyrA': 0,
    'gyrB': 0,
    'rrs': 0,
    'eis': 0,
    'embB': 0,
    'pncA': 0
}

sample_pre_xdr = {
    'rpoB': 1,
    'katG': 1,
    'inhA': 0,
    'gyrA': 1,
    'gyrB': 0,
    'rrs': 0,
    'eis': 0,
    'embB': 0,
    'pncA': 0
}

print('No resistance sample:')
print(json.dumps(predict_genomic_resistance_who_rules(sample_no_resistance), indent=2))

print('\nMDR sample:')
print(json.dumps(predict_genomic_resistance_who_rules(sample_mdr), indent=2))

print('\nPre-XDR concern sample:')
print(json.dumps(predict_genomic_resistance_who_rules(sample_pre_xdr), indent=2))

No resistance sample:
{
  "model_name": "WHO-catalogue-style genomic resistance expert",
  "model_type": "who_rule_based_catalogue_expert",
  "predicted_class_id": 0,
  "resistance_class": "No major genomic resistance marker detected",
  "confidence": "medium",
  "markers_detected": [],
  "resistance_evidence": [],
  "interpreted_resistance": {
    "rifampicin": false,
    "isoniazid": false,
    "fluoroquinolone": false,
    "second_line_injectable": false,
    "ethambutol_or_pyrazinamide": false
  },
  "class_probabilities": {
    "No major resistance marker detected": 0.85,
    "Drug-resistant TB / MDR-TB concern": 0.1,
    "pre-XDR concern / MDR-TB with fluoroquinolone resistance": 0.1
  },
  "probability_note": "This artifact is rule-based. Returned class scores are heuristic confidence scores, not calibrated ML probabilities.",
  "source_basis": "WHO Catalogue of mutations in Mycobacterium tuberculosis complex and their association with drug resistance, 2023 update.",
  "human_in

## 7. RAG text snippet to add to FAISS database

In [14]:
who_rag_text = '''
The WHO catalogue of mutations in Mycobacterium tuberculosis complex and their association with drug resistance provides a reference standard for interpreting genetic mutations associated with resistance to anti-tuberculosis medicines.

In the TB-Guard genomic expert, mutation-marker inputs are interpreted using WHO-catalogue-style resistance logic. rpoB is treated as evidence for rifampicin resistance concern. katG and inhA are treated as evidence for isoniazid resistance concern. gyrA and gyrB are treated as evidence for fluoroquinolone resistance concern. rrs and eis are treated as evidence for second-line injectable or aminoglycoside-associated resistance concern. embB is treated as evidence for ethambutol resistance concern, and pncA is treated as evidence for pyrazinamide resistance concern.

Rifampicin resistance together with isoniazid resistance raises concern for MDR-TB. MDR/RR-TB with additional fluoroquinolone resistance raises concern for pre-XDR TB. The demo avoids overclaiming XDR-TB from marker flags alone because clinical definitions and confirmatory drug-susceptibility testing are required.

Genomic findings must be interpreted with clinical context and confirmed by qualified clinicians and laboratory testing. The WHO-catalogue-style genomic module is therefore used as an interpretable decision-support component, not as an independent diagnostic authority.
'''.strip()

rag_text_path = GENOMIC_DIR / 'who_mutation_catalogue_rag_text.txt'
rag_text_path.write_text(who_rag_text, encoding='utf-8')

print('Saved RAG text:', rag_text_path)
print(who_rag_text)

Saved RAG text: tbguard_project/data/genomic/who_mutation_catalogue_rag_text.txt
The WHO catalogue of mutations in Mycobacterium tuberculosis complex and their association with drug resistance provides a reference standard for interpreting genetic mutations associated with resistance to anti-tuberculosis medicines.

In the TB-Guard genomic expert, mutation-marker inputs are interpreted using WHO-catalogue-style resistance logic. rpoB is treated as evidence for rifampicin resistance concern. katG and inhA are treated as evidence for isoniazid resistance concern. gyrA and gyrB are treated as evidence for fluoroquinolone resistance concern. rrs and eis are treated as evidence for second-line injectable or aminoglycoside-associated resistance concern. embB is treated as evidence for ethambutol resistance concern, and pncA is treated as evidence for pyrazinamide resistance concern.

Rifampicin resistance together with isoniazid resistance raises concern for MDR-TB. MDR/RR-TB with additional

## 8. Download artifacts from Colab

In [15]:
try:
    from google.colab import files
    files.download(str(MODEL_DIR / 'genomic_resistance_model.joblib'))
    files.download(str(MODEL_DIR / 'genomic_resistance_model_metadata.json'))
    files.download(str(GENOMIC_DIR / 'who_mutation_catalogue_cleaned.csv'))
    files.download(str(GENOMIC_DIR / 'who_mutation_catalogue_rag_text.txt'))
except Exception as e:
    print('Download skipped:', e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>